# 01 — Iconclass corpus and hierarchy

This notebook is the executable entry point for **Phase 1**. It demonstrates the same code paths used on the full Iconclass AI Test Set while defaulting to tiny repository fixtures.

The full 3.1 GB image archive is deliberately **not** downloaded here. The notebook first validates the public `data.json` shape, builds a canonical manifest, parses a documented branch of the open Iconclass hierarchy, and checks the hierarchy-aware relevance function and deterministic split machinery.

The important methodological caveat is explicit: the public `data.json` does not provide book/edition groups, so the item-hash split below is a diagnostic baseline, not the final leakage-controlled evaluation split.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from caypollard.evaluation import ndcg_at_k
from caypollard.datasets.iconclass import (
    audit_annotations, build_manifest, load_testset_annotations
)
from caypollard.graphs.iconclass import (
    build_parent_index, child_edges, hierarchical_similarity, parse_notations
)
from caypollard.provenance import manifest_digest
from caypollard.splitting import split_records

ROOT

## 1. Load the official `data.json` excerpt

In [ ]:
annotations = load_testset_annotations(
    ROOT / "data/samples/iconclass_testset_excerpt.json"
)
audit = audit_annotations(annotations)
audit

The excerpt contains the first four records shown on the official Iconclass AI Test Set page. The point here is format validation, not statistical inference from four images.

## 2. Build a canonical manifest

In [ ]:
manifest = build_manifest(annotations)
manifest[:2]

In [ ]:
split_manifest = split_records(manifest, seed="iconclass-v0.1")
{
    "digest": manifest_digest(split_manifest),
    "split_counts": {
        split: sum(row["split"] == split for row in split_manifest)
        for split in ("train", "validation", "test")
    },
}

`group_id` remains null because the source excerpt does not contain work/book/edition provenance. Fabricating such groups would make the benchmark look tidier while making the research worse, which is an impressively common bargain.

## 3. Parse the Iconclass hierarchy source format

In [ ]:
records = parse_notations(ROOT / "data/samples/iconclass_notations_fixture.txt")
edges = child_edges(records)
parents = build_parent_index(edges)
len(records), len(edges), edges[:8]

The fixture contains the documented path `2 → 25 → 25G → 25G4 → 25G41` plus children shown in the official Python example. A full run uses the pinned `notations.txt` from `iconclass/data`.

## 4. Graded hierarchy relevance

In [ ]:
examples = [
    ("25G41", "25G41"),
    ("25G41", "25G411"),
    ("25G411", "25G412"),
    ("25G41(+1)", "25G41"),
]
[
    {"a": a, "b": b, "similarity": hierarchical_similarity(a, b, parents)}
    for a, b in examples
]

### nDCG smoke check

The primary endpoint uses graded hierarchy relevance rather than flattening every non-identical Iconclass concept to zero.

In [ ]:
# A perfectly ordered toy ranking and a deliberately worse ranking.
{
    "ideal": ndcg_at_k([1.0, 0.5, 1/3, 0.0], 4),
    "misordered": ndcg_at_k([0.0, 1/3, 0.5, 1.0], 4),
}

This baseline is intentionally transparent: similarity is `1 / (1 + shortest hierarchy distance through a common ancestor)`. It is a benchmark definition to be stress-tested, not a metaphysical theory of iconographic meaning.

## 5. Write a smoke-test result artifact

In [ ]:
result = {
    "fixture_audit": audit,
    "manifest_digest": manifest_digest(split_manifest),
    "hierarchy_records": len(records),
    "hierarchy_edges": len(edges),
}
out = ROOT / "results/metrics/iconclass_phase1_smoke.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
out

## 6. Running the full phase-1 pipeline

After reviewing `docs/ICONCLASS_DATA_CARD.md`:

```bash
uv run python scripts/download_iconclass_testset.py --accept-large-download
uv run python scripts/fetch_iconclass_core.py
# extract the official archive outside Git-tracked paths, then:
uv run python scripts/build_iconclass_benchmark.py path/to/data.json \
  --notations data/raw/iconclass/notations.txt
```

Before any headline model comparison, add exact/near-duplicate groups and stronger provenance grouping as required by `docs/protocol-v0.1.md`.